In [1]:
%run spectrum_toolbox.py

## Multi Parameter from JSON

In [3]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def plot_shot_parameters(shot, time_range, save_dir=r'D:\TEMP'):
    time1, time2 = time_range
    save_path = Path(save_dir) / f'plasma_parameters_{shot}.png'
    failed_signals = []
    
    try:
        loader = MDSDataLoader()
        fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6)) = plt.subplots(3, 2, figsize=(12, 9))
        fig.suptitle(f'Shot #{shot}: EAST Plasma Parameters', fontsize=16, fontweight='bold', y=0.995)

        # ========== 子图1-3：使用辅助函数 ==========
        def safe_load_signal(ax, shot, signal, tree, label, ylabel, ylim=None, factor=1.0):
            try:
                time, data = loader.get_signal(shot, signal, (time1, time2), tree=tree)
                ax.plot(time, data / factor, color='red', linewidth=2, label=label)
                if ylim:
                    ax.set_ylim(ylim)
                ax.set_ylabel(ylabel)
                ax.legend(loc='best', frameon=True, shadow=True)
                return True
            except Exception as e:
                failed_signals.append(f"{signal} ({tree}): {str(e)}")
                ax.text(0.5, 0.5, f'Signal Not Available\n{signal}', 
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=12, color='red', fontweight='bold')
                ax.set_ylabel(ylabel)
                return False

        safe_load_signal(ax1, shot, '\\WMHD', 'efitrt_east', 'WMHD', 'WMHD [kJ]', ylim=(0, 250), factor=1000)
        safe_load_signal(ax2, shot, '\\Q95', 'efit_east', 'Q95', 'Q95')
        safe_load_signal(ax3, shot, '\\dfsdev', 'pcs_east', 'ne average', 'ne average [1e19 m^-3]')

        # ========== 子图4: ECRH ==========
        try:
            EC, time_ec = {}, {}
            valid_ec_count = 0
            for i in range(4):
                try:
                    time_ec[i], EC[i] = loader.get_signal(shot, f'\\ECRH{i+1}', (time1, time2), tree='east')
                    valid_ec_count += 1
                except:
                    failed_signals.append(f'\\ECRH{i+1} (east): 数据加载失败')
                    
            if valid_ec_count > 0:
                base_time = next(t for t in time_ec.values() if t is not None)
                EC_total = np.zeros_like(base_time)
                for i in range(4):
                    if i in EC and EC[i] is not None:
                        if len(time_ec[i]) != len(base_time):
                            EC_interp = np.interp(base_time, time_ec[i], EC[i])
                            EC_total += EC_interp
                        else:
                            EC_total += EC[i]
                ax4.plot(base_time, EC_total, color='red', linewidth=2, label='Total ECRH Power')
                ax4.set_ylabel('ECRH Power [MW]')
                ax4.set_ylim(0, 8)
                ax4.legend(loc='best', frameon=True, shadow=True)
            else:
                raise Exception("所有ECRH信号均缺失")
        except Exception as e:
            failed_signals.append(f"ECRH: {str(e)}")
            ax4.text(0.5, 0.5, 'All ECRH Signals\nNot Available', 
                    ha='center', va='center', transform=ax4.transAxes,
                    fontsize=12, color='red', fontweight='bold')
            ax4.set_ylabel('ECRH Power [MW]')

        # ========== 子图5: LHW功率（独立容错）==========
        LHW_failed = []
        LHW_plotted = False
        
        # LHW1
        try:
            time_lhw1, plhi1 = loader.get_signal(shot, '\\plhi1', (time1, time2), tree='east')
            time_lhr1, plhr1 = loader.get_signal(shot, '\\plhr1', (time1, time2), tree='east')
            LHW1_total = (plhi1 - plhr1) / 1000
            ax5.plot(time_lhw1, LHW1_total, color='red', linewidth=2, label='LHW1 Power')
            LHW_plotted = True
        except Exception as e:
            LHW_failed.append(f"LHW1: {str(e)}")
        
        # LHW2
        try:
            time_lhw2, plhi2 = loader.get_signal(shot, '\\plhi2', (time1, time2), tree='east')
            time_lhr2, plhr2 = loader.get_signal(shot, '\\plhr2', (time1, time2), tree='east')
            LHW2_total = (plhi2 - plhr2) / 1000
            
            plot_time = time_lhw1 if 'time_lhw1' in locals() else time_lhw2
            ax5.plot(plot_time, LHW2_total, color='blue', linewidth=2, label='LHW2 Power')
            LHW_plotted = True
        except Exception as e:
            LHW_failed.append(f"LHW2: {str(e)}")
        
        if LHW_plotted:
            ax5.set_ylabel('LHW Power [MW]')
            ax5.legend(loc='best', frameon=True, shadow=True, ncol=2)
        else:
            failed_signals.extend(LHW_failed)
            ax5.text(0.5, 0.5, 'All LHW Signals\nNot Available', 
                    ha='center', va='center', transform=ax5.transAxes,
                    fontsize=12, color='red', fontweight='bold')
            ax5.set_ylabel('LHW Power [MW]')

        # ========== 子图6: NBI功率（独立容错）==========
        NBI_failed = []
        NBI_plotted = False
        
        # Co-NBI
        try:
            time_nbi1, pnbi1l = loader.get_signal(shot, '\\PNBI1LSOURCE', (time1, time2), tree='nbi_east')
            _, pnbi1r = loader.get_signal(shot, '\\PNBI1RSOURCE', (time1, time2), tree='nbi_east')
            co_NBI_total = (pnbi1l + pnbi1r) / 1000
            ax6.plot(time_nbi1, co_NBI_total, color='red', linewidth=2, label='Co-NBI Power')
            NBI_plotted = True
        except Exception as e:
            NBI_failed.append(f"Co-NBI: {str(e)}")
        
        # Ctr-NBI
        try:
            time_nbi2, pnbi2l = loader.get_signal(shot, '\\PNBI2LSOURCE', (time1, time2), tree='nbi_east')
            _, pnbi2r = loader.get_signal(shot, '\\PNBI2RSOURCE', (time1, time2), tree='nbi_east')
            ctr_NBI_total = (pnbi2l + pnbi2r) / 1000
            
            plot_time = time_nbi1 if 'time_nbi1' in locals() else time_nbi2
            ax6.plot(plot_time, ctr_NBI_total, color='blue', linewidth=2, label='Ctr-NBI Power')
            NBI_plotted = True
        except Exception as e:
            NBI_failed.append(f"Ctr-NBI: {str(e)}")
        
        if NBI_plotted:
            ax6.set_ylabel('NBI Power [MW]')
            ax6.legend(loc='best', frameon=True, shadow=True, ncol=2)
        else:
            failed_signals.extend(NBI_failed)
            ax6.text(0.5, 0.5, 'All NBI Signals\nNot Available', 
                    ha='center', va='center', transform=ax6.transAxes,
                    fontsize=12, color='red', fontweight='bold')
            ax6.set_ylabel('NBI Power [MW]')

        plt.tight_layout()
        plt.subplots_adjust(top=0.93)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        if failed_signals:
            print(f"⚠ Shot {shot}: 部分信号缺失")
            for sig in failed_signals:
                print(f"  - {sig}")
        
        print(f"✓ Shot {shot} 处理成功")
        return True
        
    except Exception as e:
        print(f"✗ Shot {shot} 严重错误: {str(e)}")
        if 'fig' in locals():
            plt.close(fig)
        return False

# ========== 主程序：批量处理 ==========
if __name__ == '__main__':
    json_path = r'D:\MyPythonCodes\experiment_time_records.json'
    save_directory = r'D:\TEMP\shot_plots'
    Path(save_directory).mkdir(exist_ok=True)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        shots_data = json.load(f)
    
    success_count = 0
    failed_shots = []
    
    for shot_str, data in shots_data.items():
        shot_num = int(shot_str)
        time_range = data['time_range']
        
        if plot_shot_parameters(shot_num, time_range, save_dir=save_directory):
            success_count += 1
        else:
            failed_shots.append(shot_num)
    
    print(f"\n{'='*50}")
    print(f"批量处理完成: 成功 {success_count} / 总计 {len(shots_data)}")
    if failed_shots:
        print(f"失败的炮号: {failed_shots}")

⚠️ 警告：\WMHD 只有3点
⚠️ 警告：\Q95 只有1点
⚠️ 警告：\dfsdev 只有35点
✓ Shot 158900 处理成功
⚠️ 警告：\WMHD 只有41点
⚠️ 警告：\Q95 只有7点
✓ Shot 157230 处理成功
⚠️ 警告：\WMHD 只有4点
⚠️ 警告：\Q95 只有1点
⚠️ 警告：\dfsdev 只有50点
✓ Shot 157273 处理成功
⚠️ 警告：\WMHD 只有6点
⚠️ 警告：\dfsdev 只有75点
⚠ Shot 158591: 部分信号缺失
  - \Q95 (efit_east): 读取信号失败 \Q95 (tree=efit_east): 时间范围 (2.6, 2.75) 内无数据
✓ Shot 158591 处理成功
⚠️ 警告：\WMHD 只有4点
⚠️ 警告：\Q95 只有1点
⚠️ 警告：\dfsdev 只有40点
✓ Shot 158883 处理成功
⚠️ 警告：\WMHD 只有3点
⚠️ 警告：\dfsdev 只有40点
⚠ Shot 158889: 部分信号缺失
  - \Q95 (efit_east): 读取信号失败 \Q95 (tree=efit_east): 时间范围 (2.65, 2.73) 内无数据
✓ Shot 158889 处理成功
⚠️ 警告：\WMHD 只有4点
⚠️ 警告：\dfsdev 只有50点
⚠ Shot 158896: 部分信号缺失
  - \Q95 (efit_east): 读取信号失败 \Q95 (tree=efit_east): 时间范围 (2.8, 2.9) 内无数据
✓ Shot 158896 处理成功
⚠️ 警告：\WMHD 只有4点
⚠️ 警告：\Q95 只有1点
⚠️ 警告：\dfsdev 只有40点
✓ Shot 158902 处理成功
⚠️ 警告：\WMHD 只有48点
⚠️ 警告：\Q95 只有6点
✓ Shot 159092 处理成功
⚠️ 警告：\WMHD 只有49点
⚠️ 警告：\Q95 只有6点
✓ Shot 159093 处理成功
⚠️ 警告：\WMHD 只有24点
⚠️ 警告：\Q95 只有3点
✓ Shot 159317 处理成功
⚠️ 警告：\WMHD 只有25点
⚠️ 警告：\Q95 只有4点
✓ Shot 159

## Multi spectrogram point ece megnatic probe

In [2]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import numpy as np

# 线程锁防止多线程绘图冲突
plot_lock = threading.Lock()

# ===== 诊断配置 =====
DIAG_CONFIG = {
    'ece': {'channels': 9, 'start': 1, 'tree': 'ece1-9', 'rows': 9},
    'khpt': {'channels': 3, 'start': 6, 'tree': 'khpt', 'rows': 3},
    'pointn': {'channels': 11, 'start': 1, 'tree': 'pointn', 'rows': 11}
}

def calculate_spectrogram_figsize(nrows, base_height=3.5, max_height=30):
    """智能计算频谱图尺寸"""
    height = nrows * base_height * 1.1  # 增加10%间距
    height = min(height, max_height)
    width = 8 if nrows > 4 else 9  # 通道少时更宽
    return (width, height)

def plot_single_diagnostic(shot, time_range, diag_type, save_dir):
    """
    绘制单种诊断的频谱图
    diag_type: 'ece', 'khpt', 'pointn'
    """
    time1, time2 = time_range
    config = DIAG_CONFIG[diag_type]
    save_path = Path(save_dir) / diag_type / f'spectrum_{shot}_{diag_type}.png'
    save_path.parent.mkdir(parents=True, exist_ok=True)  # 创建诊断类型子文件夹
    
    failed_channels = []
    
    with plot_lock:  # 线程安全
        try:
            plotter = ProbePlotter(MDSDataLoader())
            
            # 智能布局
            figsize = calculate_spectrogram_figsize(config['rows'])
            fig, axes = plt.subplots(config['rows'], 1, figsize=figsize, clear=True)
            
            if config['rows'] == 1:
                axes = [axes]  # 统一为列表
            
            fig.suptitle(f'Shot #{shot}: {diag_type.upper()} Spectrograms', 
                        fontsize=14, fontweight='bold', y=0.995)
            plt.subplots_adjust(hspace=0.25, top=0.98, bottom=0.02)
            
            # 循环通道
            for i in range(config['channels']):
                ax = axes[i]
                channel_num = config['start'] + i
                
                try:
                    spec_data = plotter.compute_spectrogram_data(
                        shot, channel_num, config['tree'], (time1, time2), (time2-time1)/50
                    )
                    spec_data.plot(ax, freq_range=(0, 100))
                    ax.set_ylabel(f'Ch.{channel_num}', fontsize=9)
                    
                except Exception as e:
                    failed_channels.append(f"Ch.{channel_num}: {str(e)}")
                    ax.text(0.5, 0.5, f'Ch.{channel_num}\nNo Data', 
                            ha='center', va='center', transform=ax.transAxes,
                            fontsize=11, color='red', fontweight='bold')
                    ax.set_ylabel(f'Ch.{channel_num}', fontsize=9)
                
                # 只显示最底部子图的x轴标签
                if i == config['rows'] - 1:
                    ax.set_xlabel('Time [s]', fontsize=10)
            
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            
            # 打印结果
            if failed_channels:
                print(f"  ⚠ {diag_type}: {len(failed_channels)}/{config['channels']} 通道失败")
            
            return True
            
        except Exception as e:
            print(f"  ✗ {diag_type} 致命错误: {e}")
            if 'fig' in locals():
                plt.close(fig)
            return False

def process_all_diagnostics(shot, time_range, save_dir):
    """处理单个炮号的所有三种诊断"""
    results = {}
    for diag_type in DIAG_CONFIG.keys():
        results[diag_type] = plot_single_diagnostic(shot, time_range, diag_type, save_dir)
    return shot, results


# ========== 主程序：多线程批量处理 ==========
if __name__ == '__main__':
    json_path = r'D:\MyPythonCodes\experiment_time_records.json'
    save_directory = r'D:\TEMP\shot_spectra'
    
    # 读取JSON
    with open(json_path, 'r', encoding='utf-8') as f:
        shots_data = json.load(f)
    
    # 创建任务列表（所有炮号×所有诊断类型）
    tasks = []
    for shot_str, data in shots_data.items():
        shot_num = int(shot_str)
        time_range = data['time_range']
        for diag_type in DIAG_CONFIG.keys():
            tasks.append((shot_num, time_range, diag_type, save_directory))
    
    # 多线程并行处理（建议max_workers=4-8）
    max_workers = min(6, len(tasks))
    success_count = 0
    
    print(f"开始处理 {len(tasks)} 个任务 (炮号×诊断类型)...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {
            executor.submit(plot_single_diagnostic, *task): task
            for task in tasks
        }
        
        for future in as_completed(future_to_task):
            shot, time_range, diag_type, _ = future_to_task[future]
            try:
                if future.result():
                    success_count += 1
                    print(f"✓ Shot {shot} {diag_type.upper()} 完成")
                else:
                    print(f"✗ Shot {shot} {diag_type.upper()} 失败")
            except Exception as e:
                print(f"✗ Shot {shot} {diag_type.upper()} 异常: {e}")
    
    print(f"\n{'='*60}")
    print(f"全部处理完成: 成功 {success_count} / 总计 {len(tasks)}")
    print(f"图片保存在: {Path(save_directory).resolve()}")

开始处理 138 个任务 (炮号×诊断类型)...
✓ Shot 158900 ECE 完成
✓ Shot 158900 KHPT 完成
✓ Shot 158900 POINTN 完成
✓ Shot 157230 ECE 完成
✓ Shot 157230 KHPT 完成
✓ Shot 157230 POINTN 完成
✓ Shot 157273 ECE 完成
✓ Shot 157273 KHPT 完成
✓ Shot 157273 POINTN 完成
✓ Shot 158591 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 158591 KHPT 完成
✓ Shot 158591 POINTN 完成
✓ Shot 158883 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 158883 KHPT 完成
✓ Shot 158883 POINTN 完成
✓ Shot 158889 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 158889 KHPT 完成
✓ Shot 158889 POINTN 完成
✓ Shot 158896 ECE 完成
✓ Shot 158896 KHPT 完成
✓ Shot 158896 POINTN 完成
✓ Shot 158902 ECE 完成
✓ Shot 158902 KHPT 完成
✓ Shot 158902 POINTN 完成
✓ Shot 159092 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 159092 KHPT 完成
✓ Shot 159092 POINTN 完成
✓ Shot 159093 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 159093 KHPT 完成
✓ Shot 159093 POINTN 完成
✓ Shot 159317 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 159317 KHPT 完成
✓ Shot 159317 POINTN 完成
✓ Shot 159328 ECE 完成
  ⚠ khpt: 3/3 通道失败
✓ Shot 159328 KHPT 完成
✓ Shot 159328 POINTN 完成
✓ Shot 159371 ECE 完成
  ⚠ khpt: 3/3 通道